In [4]:
import yaml, os, sys
from itertools import product
import pathlib as PATH
from tqdm.notebook import tqdm
import copy

def main(config, list_inner_edges, list_outer_edges, special_cases=[], doextract=True,
         do_fit_mass=True, do_fit_correl=True, do_produce_final=True, do_prompt_only=False):
    def run_command(cmd, log, label):
        log.parent.mkdir(parents=True, exist_ok=True)
        print(f"[{label}] {cmd} > {log} 2>&1")
        out = os.system(f"{cmd} > {log} 2>&1")
        if out != 0:
            print(f"[{label}] WARNING: Command exited with code {out}")

    def task_modify_config(config_path, inner_edge, outer_edge, task_LM=None):
        with open(config_path) as f:
            config = copy.deepcopy(yaml.safe_load(f))
        config['deltaEtaBins'] = [
            [-float(outer_edge), -float(inner_edge)],
            [float(inner_edge), float(outer_edge)]
        ]
        if config.get("method") == "MassBinning":
            config['rebinDeltaPhi'] = 4
        if task_LM is not None:
            config['pathFileSE'] = task_LM['pathFileSE']
            config['pathFileME'] = task_LM['pathFileME']
            config['pathFileMass'] = task_LM['pathFileMass']
            config['outdir'] = task_LM['outdir']
            # config['nDeltaPhiBins'] = task_LM['nDeltaPhiBins']
            config['nDeltaPhiBins'] = 16
            if config.get("method") == "MassBinning":
                config['rebinDeltaPhi'] = 4

        # generate suffix from edges only for non-trial configs (preserve trial_{id} for sys trials)
        if not config.get('suffix', '').startswith('trial_'):
            config['suffix'] = f'{inner_edge.replace(".", "d")}_{outer_edge.replace(".", "d")}'

        output_dir = PATH.Path(config['outdir']) / f"CorrelExtract_{config['suffix']}"
        output_dir.mkdir(parents=True, exist_ok=True)
        out_config_path = output_dir / config_path.name.replace('.yaml', f'_{config["suffix"]}.yaml')
        with open(out_config_path, 'w') as f:
            yaml.dump(config, f, default_flow_style=False)
        print(f"Generated config file: {out_config_path}")
        return out_config_path

    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
    print(f"[INFO] Project root: {PROJECT_ROOT}")
    PROJECT_ROOT = "/home/wuct/ALICE/reps/hf-vn-dev/dev"

    def task_extract_correl(config_path):
        run_command(f"cd {PROJECT_ROOT}/correlations/PostProcessing && python3 {PROJECT_ROOT}/correlations/PostProcessing/ExtractOutputCorrel.py {config_path}",
                    config_path.parent / f"log_extract_{config_path.stem}.txt", "Extract")

    def task_fit_mass(config_path):
        run_command(f"cd {PROJECT_ROOT}/src && python3 {PROJECT_ROOT}/src/ry_interface.py {config_path}",
                    config_path.parent / f"log_fit_mass_{config_path.stem}.txt", "FitMass")

    def task_fit_correl(config_path):
        run_command(f"cd {PROJECT_ROOT}/correlations/PostProcessing && python3 {PROJECT_ROOT}/correlations/PostProcessing/FitCorrel.py {config_path}",
                    config_path.parent / f"log_fit_correl_{config_path.stem}.txt", "FitCorrel")

    def task_extract_ry_trigger(config_path):
        run_command(f"cd {PROJECT_ROOT}/correlations/PostProcessing && python3 {PROJECT_ROOT}/correlations/PostProcessing/construct_v2_mass.py extract-ry-trigger {config_path}",
                    config_path.parent / f"log_ry_trigger_{config_path.stem}.txt", "RyTrig")

    def task_construct_v2_mass(config_path):
        run_command(f"cd {PROJECT_ROOT}/correlations/PostProcessing && python3 {PROJECT_ROOT}/correlations/PostProcessing/construct_v2_mass.py build-mass-v2 {config_path}",
                    config_path.parent / f"log_construct_v2_mass_{config_path.stem}.txt", "MassV2")

    def task_simfit(config_path):
        with open(config_path) as f:
            config = yaml.safe_load(f)
        if config.get("method") != "MassBinning":
            print("[SimFit] Skipping since method is not MassBinning")
            return
        outdir = PATH.Path(config['outdir'])
        suffix = config['suffix']
        mass_v2_dir = outdir / f"CorrelExtract_{suffix}" / "CorrelationFitResults" / "MassVsV2"
        pt_bins_cand = [float(x) for x in config['ptBinsCand']]
        pt_bins_had = [float(x) for x in config['ptBinsHad']]
        fit_config = config.get('fitConfig', {})
        Dmeson = config.get("Dmeson", "Dzero")
        get_vn_script = f"{PROJECT_ROOT}/src/get_vn_vs_mass.py"
        if not os.path.exists(get_vn_script):
            print("[SimFit] WARNING: get_vn_vs_mass.py not found, skipping SimFit")
            return
        simfit_config = {
            "Dmeson": Dmeson,
            "ptbins": pt_bins_cand,
            "centrality": config.get("centrality", "k020"),
            "v2extraction": {
                "SgnFunc": fit_config.get("SgnFunc", "kGaus"),
                "BkgFunc": fit_config.get("BkgFunc", "kExpo"),
                "BkgFuncVn": fit_config.get("BkgFuncVn", "kLin"),
                "Sigma": fit_config.get("Sigma", [0.02] * (len(pt_bins_cand)-1)),
                "MassFitRanges": fit_config.get("MassFitRanges", [[pt_bins_cand[0], pt_bins_cand[-1]]] * (len(pt_bins_cand)-1)),
                "Rebin": fit_config.get("Rebin", 4),
                "FixSigma": fit_config.get("FixSigma", 0),
                "FixMean": fit_config.get("FixMean", 0),
                "InclSecPeak": fit_config.get("InclSecPeak", 0),
                "enableRef": fit_config.get("enableRef", False),
                "ReflFunc": fit_config.get("ReflFunc", "2gaus")
            }
        }
        for i_pt_had in range(len(pt_bins_had)-1):
            pt_had_label = f"PtAssoc{int(pt_bins_had[i_pt_had]*10):02d}to{int(pt_bins_had[i_pt_had+1]*10):02d}"
            mass_v2_file = mass_v2_dir / f"InvMassVsV2_{pt_had_label}.root"
            if not mass_v2_file.exists():
                print(f"[SimFit] WARNING: {mass_v2_file} not found, skipping {pt_had_label}")
                continue
            simfit_config_file = mass_v2_dir / f"simfit_config_{pt_had_label}.yaml"
            with open(simfit_config_file, 'w') as f:
                yaml.dump(simfit_config, f, default_flow_style=False)
            log_file = outdir / f"CorrelExtract_{suffix}" / f"log_simfit_{pt_had_label}.txt"
            print(f"[SimFit] Running SimFit for {pt_had_label}...")
            os.system(f"python3 {get_vn_script} {simfit_config_file} {mass_v2_file} -b > {log_file} 2>&1")

    import ROOT
    def get_final_results_path(config_path):
        with open(config_path) as f:
            config = yaml.safe_load(f)
        outdir = PATH.Path(config['outdir'])
        suffix = config['suffix']
        return outdir / f"CorrelExtract_{suffix}" / "final_results.root"

    def produce_final_results(config_path):
        with open(config_path) as f:
            config = yaml.safe_load(f)
        if config.get("method") != "DeltaPhiBinning":
            print("[FinalResults] Skipping since method is not DeltaPhiBinning")
            return
        outdir = PATH.Path(config['outdir'])
        suffix = config['suffix']
        input_file = outdir / f"CorrelExtract_{suffix}" / "CorrelationFitResults" / "Output_CorrelationFitting_Root" / "CorrPhiD0_FinalPlots.root"
        output_file = get_final_results_path(config_path)
        print(f"[FinalResults] Processing {input_file} -> {output_file}")
        if not input_file.exists():
            print(f"[FinalResults] WARNING: Input file {input_file} does not exist, skipping")
            return
        f = ROOT.TFile.Open(str(input_file))
        for k in f.GetListOfKeys():
            print(f"[FinalResults] Processing histogram: {k.GetName()}")
            h = f.Get(k.GetName())
            h.SetDirectory(0)
            h.Scale(1/0.07)
            o = ROOT.TFile.Open(str(output_file), "UPDATE")
            o.cd()
            h.Write(k.GetName(), ROOT.TObject.kOverwrite)
            o.Close()
        f.Close()
        return output_file

    def calculate_prompt_v2(final_results_file, prompt_config=None):
        if prompt_config is None:
            prompt_config = f"{PROJECT_ROOT}/configs/v2_prompt_v2_method_check.yml"
        run_command(f"cd {PROJECT_ROOT}/src && python3 {PROJECT_ROOT}/src/compute_prompt_v2_unfold.py {prompt_config} {final_results_file} --outpath {PATH.Path(final_results_file).parent}",
                    PATH.Path(final_results_file).parent / f"log_compute_prompt_v2.txt", "ComputePromptV2")

    # ===== MAIN LOOP =====
    # config = f"{PROJECT_ROOT}/correlations/PostProcessing/config_CorrAnalysis_v2_010_negDeta.yaml"
    # config = f"/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/CorrelExtract_0d2_1d3_AppDeltaPhi/results_fixed_noBL_tempFuncGausPeriodic/config_CorrAnalysis_v2_010_negDeta_0d2_1d3_AppDeltaPhi.yaml"
    # config = f"/home/wuct/ALICE/reps/hf-vn-dev/dev/correlations/PostProcessing/config_CorrAnalysis_v2_010_negDeta.yaml"
    # config = f"/home/wuct/ALICE/reps/hf-vn-dev/dev/correlations/PostProcessing/config_CorrAnalysis_v2_20_50.yaml"
    # config = f"/home/wuct/ALICE/reps/hf-vn-dev/dev/correlations/PostProcessing/config_CorrAnalysis_v2_010_negDeta.yaml"
    # list_inner_edges = ["0.2", "0.3", "0.4", "0.5", "0.6"]
    # list_inner_edges = ["0.2"]
    # list_outer_edges = ["1.3"]
    # special_cases = []

    with open(config, 'r') as f:
        base_config = yaml.safe_load(f)
    method = base_config.get("method", "DeltaPhiBinning")
    print(f"[INFO] Using method: {method}")

    # for inner_edge, outer_edge in tqdm(list(product(list_inner_edges, list_outer_edges)) + special_cases,
    #                                     desc="Varying delta eta ranges", unit="config pairs"):
    for inner_edge, outer_edge in list(product(list_inner_edges, list_outer_edges)) + special_cases:
        if float(outer_edge) <= float(inner_edge) + 0.1:
            print(f"[INFO] Skipping invalid config: inner_edge={inner_edge}, outer_edge={outer_edge}")
            continue
        print(f"\n{'='*60}\n  inner={inner_edge}, outer={outer_edge}\n{'='*60}")
        out_config_path = task_modify_config(PATH.Path(config), inner_edge, outer_edge)
        
        # === prompt-only mode: skip all other steps, only recompute prompt v2 ===
        if do_prompt_only:
            final_results_file = get_final_results_path(out_config_path)
            if not final_results_file.exists():
                print(f"[Prompt] WARNING: {final_results_file} not found, skipping prompt calc")
                continue
            calculate_prompt_v2(final_results_file)
            continue
        
        # === Step 1: Extract correlations ===
        print(">>> Step 1: Extract")
        if doextract:
            task_extract_correl(out_config_path)
        else:
            print("[INFO] Skipping extraction step as per doextract=False")

        if do_fit_mass:
            if method == "MassBinning":
                print(">>> Step 2a: Extract ry_trigger")
                task_extract_ry_trigger(out_config_path)
            if method == "DeltaPhiBinning":
                print(">>> Step 2: Fit mass")
                task_fit_mass(out_config_path)
        
        # === LM template ===
        if 'task_LM' in base_config and base_config['task_LM'].get('do', False):
            out_config_path_lm = task_modify_config(PATH.Path(config), inner_edge, outer_edge, base_config['task_LM'])
            print(">>> LM Template: Extract correlations")
            if doextract:
                task_extract_correl(out_config_path_lm)
            else:
                print("[INFO] Skipping extraction step as per doextract=False")
            if do_fit_mass:
                if method == "MassBinning":
                    print(">>> LM Template: Extract ry_trigger")
                    task_extract_ry_trigger(out_config_path_lm)
                if method == "DeltaPhiBinning":
                    print(">>> LM Template: Fit mass")
                    print(f">>> out_config_path_lm: {out_config_path_lm}")
                    task_fit_mass(out_config_path_lm)

        #== Step 3: Fit correlations ===
        if do_fit_correl:
            print(">>> Step 3: Fit correlations")
            task_fit_correl(out_config_path)
        
        # === MassBinning-only steps ===
        if method == "MassBinning":
            print(">>> Step 4: Build v2-vs-mass")
            task_construct_v2_mass(out_config_path)
            print(">>> Step 5: Simultaneous fit")
            task_simfit(out_config_path)
        
        # === DeltaPhiBinning-only: produce final results ===
        final_results_file = None
        if method == "DeltaPhiBinning":
            if do_produce_final:
                print(">>> Final: Produce final results")
                final_results_file = produce_final_results(out_config_path)
            else:
                final_results_file = get_final_results_path(out_config_path)
        if final_results_file is not None:
            calculate_prompt_v2(final_results_file)

    print("\n[INFO] All done!")

In [5]:
# import copy
from itertools import product
import pathlib as PATH

def modify_config_sys(config, trial_config):
    with open(config, 'r') as f:
        config = yaml.safe_load(f)
    sys_outdir = PATH.Path(config.get('sys_outdir'))
    mother_config = copy.deepcopy(config)
    
    config_trials = {} # output
    
    with open(trial_config, 'r') as f:
        trial_params = yaml.safe_load(f)

    trials = {}
    iPt = 0

    nDeltaPhiBins = trial_params['nDeltaPhiBins']

    for pt_trial in trial_params['pTBins']:
        massMins = pt_trial['multitrial']['MassMin']
        massMaxs = pt_trial['multitrial']['MassMax']
        rebins = pt_trial['multitrial']['Rebin']
        bkgFuncs = pt_trial['multitrial']['BkgFunc']

        for pt_range in pt_trial['ranges']:
            for i_massMin, i_massMax, i_rebin, i_bkgFunc, i_nDeltaPhi in product(
                range(len(massMins)),
                range(len(massMaxs)),
                range(len(rebins)),
                range(len(bkgFuncs)),
                range(len(nDeltaPhiBins))
            ):
                massMin = massMins[i_massMin]
                massMax = massMaxs[i_massMax]
                rebin = rebins[i_rebin]
                bkgFunc = bkgFuncs[i_bkgFunc]
                nDeltaPhiBin = nDeltaPhiBins[i_nDeltaPhi]
                
                trial_code = f"{i_massMin+1}{i_massMax+1}{i_rebin+1}{i_bkgFunc+1}{i_nDeltaPhi+1}"

                id = trial_code
                
                if iPt == 0:
                    trial = {}
                    trial['MassFitRanges'] = [[massMin, massMax]]
                    trial['Rebin'] = [rebin]
                    trial['BkgFunc'] = bkgFunc 
                    trial['nDeltaPhiBin'] = nDeltaPhiBin 
                    trials[id] = trial
                else: 
                    if id in trials:
                        trial = trials[id]
                        trial['MassFitRanges'].append([massMin, massMax])
                        trial['Rebin'].append(rebin)
                        
            iPt += 1

    # Create new configurations for each trial
    for trial_id, trial in trials.items():
        new_config = copy.deepcopy(mother_config)
        new_config['fitConfig']['MassFitRanges'] = trial['MassFitRanges']
        new_config['fitConfig']['Rebin'] = trial['Rebin']
        new_config['fitConfig']['BkgFunc'] = trial['BkgFunc']
        new_config['nDeltaPhiBins'] = trial['nDeltaPhiBin']
        new_config['outdir'] = str(sys_outdir / f"sys_fit")
        new_config['task_LM']['outdir'] = str(sys_outdir / f"LM_sys_fit")
        new_config['suffix'] = f"trial_{trial_id}"
        new_config_path = sys_outdir / f"sys_fit" / f"config" / f"config_{trial_id}.yaml"
        new_config_path.parent.mkdir(parents=True, exist_ok=True)
        with open(new_config_path, 'w') as f:
            yaml.dump(new_config, f, default_flow_style=False)
        config_trials[trial_id] = new_config_path

    return config_trials

def reset_config(ROOT_OUT, PROJECT_ROOT, base_name="config_CorrAnalysis_v2_010_sys.yaml"):
    """Reset the results-root paths (outdir / sys_outdir / task_LM.outdir / pathFile*)
    in the base config from the old root to ROOT_OUT, then regenerate every trial
    config so they all point under ROOT_OUT. PROJECT_ROOT locates the config files.
    Returns {trial_id: config_path}."""
    cfg_dir = PATH.Path(PROJECT_ROOT) / "correlations" / "PostProcessing"
    base_cfg_path = cfg_dir / base_name
    trial_cfg_path = cfg_dir / "config_sys_trails.yaml"
    ROOT_OUT = str(PATH.Path(ROOT_OUT)).rstrip("/")

    with open(base_cfg_path) as f:
        base = yaml.safe_load(f)
    s = base.get("sys_outdir", "")
    i = s.rfind("/fifth")
    assert i >= 0, f"[reset_config] cannot auto-detect old results root from sys_outdir={s!r}"
    old_root = s[:i + len("/fifth")]

    if old_root != ROOT_OUT:
        raw = PATH.Path(base_cfg_path).read_text()
        PATH.Path(base_cfg_path).write_text(raw.replace(old_root, ROOT_OUT))
        print(f"[reset_config] base config: {old_root}\n                -> {ROOT_OUT}")
    else:
        print(f"[reset_config] base config already under {ROOT_OUT}")

    config_trials = modify_config_sys(str(base_cfg_path), str(trial_cfg_path))
    with open(base_cfg_path) as f:
        new_base = yaml.safe_load(f)
    print(f"[reset_config] regenerated {len(config_trials)} trial configs under {new_base['sys_outdir']}/sys_fit/config")
    return config_trials

In [6]:
import shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
# run the central value first, then run the systematic variations
config = "/home/wuct/ALICE/reps/hf-vn-dev/dev/correlations/PostProcessing/config_CorrAnalysis_v2_010_sys.yaml"
trial_config = "/home/wuct/ALICE/reps/hf-vn-dev/dev/correlations/PostProcessing/config_sys_trails.yaml"

# set do_prompt_only=True to skip extraction/fitting and only re-run the prompt v2 calc
do_prompt_only = False

# central value
main(config, list_inner_edges=["0.2"], list_outer_edges=["1.3"], special_cases=[], doextract=False, 
        do_fit_mass=False, do_fit_correl=False, do_produce_final=True, do_prompt_only=do_prompt_only)
sys.exit(0)
config_trials = modify_config_sys(config, trial_config)

def process_trial_outputs(trial_id, trial_config_path, CR_path, InvMassVsPt_path, LM_CR_path, LM_InvMassVsPt_path, CR32_path):
    with open(trial_config_path, 'r') as f:
        trial_config = yaml.safe_load(f)
    trial_outdir = trial_config.get('outdir')
    trial_suffix = trial_config['suffix']
    trial_lm_outdir = trial_config['task_LM']['outdir']

        # 1) main CorrelationsResults.root (16 or 32 bins)
    trial_CR_path = PATH.Path(trial_outdir) / f"CorrelExtract_{trial_suffix}" / "CorrelationsResults" / "CorrelationsResults.root"
    os.makedirs(trial_CR_path.parent, exist_ok=True)
    src_CR = CR32_path if (CR32_path is not None and trial_id.endswith("2")) else CR_path
    if src_CR != trial_CR_path:
        shutil.copy(src_CR, trial_CR_path)

        # 2) main InvMassVsPt.root (shared destination, nDeltaPhiBins-independent)
    trial_InvMassVsPt_path = PATH.Path(trial_outdir) / "InvMass" / "InvMassVsPt.root"
    os.makedirs(trial_InvMassVsPt_path.parent, exist_ok=True)
    if not trial_InvMassVsPt_path.exists():
        shutil.copy(InvMassVsPt_path, trial_InvMassVsPt_path)

        # 3) LM CorrelationsResults.root (per trial, suffix mapping)
    trial_LM_CR_path = PATH.Path(trial_lm_outdir) / f"CorrelExtract_{trial_suffix}" / "CorrelationsResults" / "CorrelationsResults.root"
    os.makedirs(trial_LM_CR_path.parent, exist_ok=True)
    shutil.copy(LM_CR_path, trial_LM_CR_path)

        # 4) LM InvMassVsPt.root (shared destination)
    trial_LM_InvMassVsPt_path = PATH.Path(trial_lm_outdir) / "InvMass" / "InvMassVsPt.root"
    os.makedirs(trial_LM_InvMassVsPt_path.parent, exist_ok=True)
    if not trial_LM_InvMassVsPt_path.exists():
        shutil.copy(LM_InvMassVsPt_path, trial_LM_InvMassVsPt_path)

    print(f"\n{'='*60}\n  Running systematic trial: {trial_id}\n{'='*60}")
    main(trial_config_path, list_inner_edges=["0.2"], list_outer_edges=["1.3"], special_cases=[], doextract=False, # it has to be False
             do_fit_mass=True, do_fit_correl=True, do_produce_final=True, do_prompt_only=do_prompt_only)

if do_prompt_only:
    # prompt-only mode: only re-run the prompt v2 calc for each trial
    for trial_id, trial_config_path in tqdm(config_trials.items(), desc="Prompt-only systematic trials", unit="trial"):
        print(f"\n{'='*60}\n  Running prompt-only systematic trial: {trial_id}\n{'='*60}")
        main(trial_config_path, list_inner_edges=["0.2"], list_outer_edges=["1.3"], special_cases=[], doextract=False, do_prompt_only=True)
else:
    # full mode: reuse the extraction outputs from the central value
    with open(config, 'r') as f:
        central_config = yaml.safe_load(f)
    central_outdir = central_config.get('outdir')
    central_suffix = central_config['suffix']
    central_lm_outdir = central_config['task_LM']['outdir']

    # central source files
    CR_path = PATH.Path(central_outdir) / f"CorrelExtract_{central_suffix}" / "CorrelationsResults" / "CorrelationsResults.root"
    InvMassVsPt_path = PATH.Path(central_outdir) / "InvMass" / "InvMassVsPt.root"
    LM_CR_path = PATH.Path(central_lm_outdir) / f"CorrelExtract_{central_suffix}" / "CorrelationsResults" / "CorrelationsResults.root"
    LM_InvMassVsPt_path = PATH.Path(central_lm_outdir) / "InvMass" / "InvMassVsPt.root"

    for label, p in [("CR", CR_path), ("InvMassVsPt", InvMassVsPt_path),
                     ("LM_CR", LM_CR_path), ("LM_InvMassVsPt", LM_InvMassVsPt_path)]:
        if not p.exists():
            print(f"[ERROR] Central value {label} not found at {p}, cannot proceed with systematic variations.")
            sys.exit(1)

    # --- run the 32-bin extraction ONCE, then reuse for all nDeltaPhiBins=32 trials ---
    PROJECT_ROOT = "/home/wuct/ALICE/reps/hf-vn-dev/dev"
    rep32_id = next((tid for tid in config_trials if tid.endswith("2")), None)
    CR32_path = None
    if rep32_id is not None:
        rep32_config_path = config_trials[rep32_id]
        with open(rep32_config_path, 'r') as f:
            rep32_config = yaml.safe_load(f)
        # Fix: apply the same deltaEtaBins as task_modify_config (inner/outer edges),
        # otherwise this direct call uses the base-config [0.8, 1.5] instead of [0.2, 1.3].
        rep32_config['deltaEtaBins'] = [[-1.3, -0.2], [0.2, 1.3]]
        with open(rep32_config_path, 'w') as f:
            yaml.dump(rep32_config, f, default_flow_style=False)
        CR32_path = PATH.Path(rep32_config['outdir']) / f"CorrelExtract_{rep32_config['suffix']}" / "CorrelationsResults" / "CorrelationsResults.root"
        log_extract32 = PATH.Path(rep32_config_path).parent / "log_extract_32bin.txt"
        cmd = (f"cd {PROJECT_ROOT}/correlations/PostProcessing && "
            f"python3 {PROJECT_ROOT}/correlations/PostProcessing/ExtractOutputCorrel.py {rep32_config_path}")
        print(f"[32bin] Running one-time 32-bin extraction: {cmd} > {log_extract32} 2>&1")
        os.system(f"{cmd} > {log_extract32} 2>&1")

        CR32_path = PATH.Path(rep32_config['outdir']) / f"CorrelExtract_{rep32_config['suffix']}" / "CorrelationsResults" / "CorrelationsResults.root"
        if not CR32_path.exists():
            print(f"[ERROR] 32-bin CorrelationsResults.root not found at {CR32_path}, cannot proceed.")
            sys.exit(1)
        print(f"[32bin] 32-bin CorrelationsResults.root ready at {CR32_path}")

    with ThreadPoolExecutor(max_workers=1) as executor:
        # submit tasks and keep track of futures
        futures = {
            executor.submit(
                process_trial_outputs, 
                trial_id, trial_config_path, 
                CR_path, InvMassVsPt_path, LM_CR_path, LM_InvMassVsPt_path, CR32_path
            ): trial_id 
            for trial_id, trial_config_path in config_trials.items()
        }
        
        # track actual completion with as_completed
        for future in tqdm(as_completed(futures), total=len(futures), desc="Systematic trials", unit="trial"):
            trial_id = futures[future]
            try:
                future.result() # retrieve result or raise exceptions if any occurred
            except Exception as e:
                print(f"[ERROR] Trial {trial_id} failed: {e}")


[INFO] Project root: /home/wuct/ALICE/reps/hf-vn-dev/dev
[INFO] Using method: DeltaPhiBinning

  inner=0.2, outer=1.3
Generated config file: /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/k60100_loose2to2d5_d20/sys/central/CorrelExtract_0d2_1d3/config_CorrAnalysis_v2_010_sys_0d2_1d3.yaml
>>> Step 1: Extract
[INFO] Skipping extraction step as per doextract=False
Generated config file: /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/k60100_loose2to2d5_d20/sys/LM/CorrelExtract_0d2_1d3/config_CorrAnalysis_v2_010_sys_0d2_1d3.yaml
>>> LM Template: Extract correlations
[INFO] Skipping extraction step as per doextract=False
>>> Final: Produce final results
[FinalResults] Processing /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/k60100_loose2to2d5_d20/sys/central/CorrelExtract_0d2_1d3/CorrelationFitResults/Output_CorrelationFitting_Root/CorrPhiD0_FinalPlots.root -> /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/k6010

SystemExit: 0

In [ ]:
import shutil
import copy
import pathlib as PATH

# ===== Variation: central LM + per-trial HM (correlation fit only, reuse mass fits) =====
# HM (main task) mass fit   -> reuse the trial's PairYieldsVsPhi.root (trial fitConfig)
# LM (task_LM template)     -> reuse the CENTRAL PairYieldsVsPhi.root (central fitConfig)
# Re-run only: FitCorrel + produce_final_results + prompt v2

# central LM mass-fit result (the LM template at its central value)
with open(config, 'r') as f:
    central_cfg = yaml.safe_load(f)
central_LM_pair = (PATH.Path(central_cfg['task_LM']['outdir'])
                   / f"CorrelExtract_{central_cfg['suffix']}"
                   / "AssociatedPairsYields" / "PairYieldsVsPhi.root")
assert central_LM_pair.exists(), f"[ERROR] central LM PairYieldsVsPhi not found: {central_LM_pair}"
print(f"[INFO] central LM template: {central_LM_pair}")

def process_trial_LM(trial_id, trial_config_path):
    with open(trial_config_path, 'r') as f:
        tc = yaml.safe_load(f)

    # new config: same trial, suffix prefixed with LM_
    new = copy.deepcopy(tc)
    new['suffix'] = f"trial_LM_{trial_id}"
    new_config_path = PATH.Path(new['outdir']) / "config" / f"config_LM_{trial_id}.yaml"
    new_config_path.parent.mkdir(parents=True, exist_ok=True)
    with open(new_config_path, 'w') as f:
        yaml.dump(new, f, default_flow_style=False)

    # 1) reuse the trial HM mass-fit result at the trial_LM suffix
    trial_HM_pair = (PATH.Path(tc['outdir']) / f"CorrelExtract_{tc['suffix']}"
                     / "AssociatedPairsYields" / "PairYieldsVsPhi.root")
    new_HM_pair = (PATH.Path(new['outdir']) / f"CorrelExtract_{new['suffix']}"
                   / "AssociatedPairsYields" / "PairYieldsVsPhi.root")
    assert trial_HM_pair.exists(), f"[ERROR] trial HM PairYieldsVsPhi not found: {trial_HM_pair}"
    new_HM_pair.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(trial_HM_pair, new_HM_pair)

    # 2) reuse the CENTRAL LM mass-fit result as the LM template at the trial_LM suffix
    new_LM_pair = (PATH.Path(new['task_LM']['outdir']) / f"CorrelExtract_{new['suffix']}"
                   / "AssociatedPairsYields" / "PairYieldsVsPhi.root")
    new_LM_pair.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(central_LM_pair, new_LM_pair)

    print(f"\n{'='*60}\n  Running trial_LM: {trial_id}\n{'='*60}")
    main(new_config_path, list_inner_edges=["0.2"], list_outer_edges=["1.3"], special_cases=[],
         doextract=False, do_fit_mass=False, do_fit_correl=True, do_produce_final=True,
         do_prompt_only=False)

for trial_id, trial_config_path in tqdm(config_trials.items(),
                                        desc="trial_LM (central LM + trial HM)", unit="trial"):
    process_trial_LM(trial_id, trial_config_path)

print("\n[INFO] trial_LM variations done!")


In [ ]:
# ===== Unified HM/LM combination pipeline =====
# Every correlation fit = original trial code + (HM, LM) binning combination:
#   suffix: trial_<code>_H<hm>L<lm>
# Asymmetric combos: LM template scaled by lm_bins/hm_bins
#   H32L16 -> 0.5 (the old "biased 32" correction, now a general rule)
#   H16L32 -> 2.0 ; symmetric (H16L16 / H32L32) -> 1.0 (plain copy)
# Only hPairsYields_vs_DeltaPhi is scaled; ry_trigger untouched -> F unchanged.

import ROOT
ROOT.gROOT.SetBatch(True)

PROJECT_ROOT = "/home/wuct/ALICE/reps/hf-vn-dev/dev"
RUN_COMBOS = ["H16L16", "H16L32", "H32L16", "H32L32"]   # subset to run now

def run_cmd(cmd, log):
    log.parent.mkdir(parents=True, exist_ok=True)
    print(f"  [RUN] {cmd} > {log} 2>&1")
    rc = os.system(f"{cmd} > {log} 2>&1")
    if rc != 0:
        print(f"  [WARN] exit code {rc}")
    return rc

def scale_pairyields(src, dst, scale=1.0, key_sub="hPairsYields_vs_DeltaPhi"):
    """Copy PairYieldsVsPhi.root; if scale != 1, scale the `key_sub` histograms."""
    fin = ROOT.TFile.Open(str(src))
    if not fin or fin.IsZombie():
        raise RuntimeError(f"cannot open {src}")
    fout = ROOT.TFile.Open(str(dst), "RECREATE")
    def rec(d, out_d):
        for k in d.GetListOfKeys():
            o = d.Get(k.GetName())
            if o is None:
                continue
            if o.InheritsFrom("TDirectoryFile"):
                out_d.mkdir(k.GetName())
                rec(o, out_d.GetDirectory(k.GetName()))
                out_d.cd()
            else:
                if o.InheritsFrom("TH1") and key_sub in k.GetName() and scale != 1.0:
                    o.SetDirectory(0)
                    o.Scale(scale)
                out_d.cd()
                o.Write(k.GetName())
    rec(fin, fout)
    fout.Close(); fin.Close()

def stage_hm_pair(tc, new_suffix):
    src = PATH.Path(tc['outdir']) / f"CorrelExtract_{tc['suffix']}" / "AssociatedPairsYields" / "PairYieldsVsPhi.root"
    dst = PATH.Path(tc['outdir']) / f"CorrelExtract_{new_suffix}" / "AssociatedPairsYields" / "PairYieldsVsPhi.root"
    if not src.exists():
        raise RuntimeError(f"HM PairYieldsVsPhi not found: {src}")
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(src, dst)
    return dst

def write_variant_config(tc, new_suffix):
    new = copy.deepcopy(tc)
    new['suffix'] = new_suffix
    out = PATH.Path(tc['outdir']) / "config" / f"config_{new_suffix}.yaml"
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, 'w') as f:
        yaml.dump(new, f, default_flow_style=False)
    return out

def fit_correl_only(config_path):
    p = PATH.Path(config_path)
    run_cmd(f"cd {PROJECT_ROOT}/correlations/PostProcessing && python3 {PROJECT_ROOT}/correlations/PostProcessing/FitCorrel.py {config_path}",
            p.parent / f"log_fit_correl_{p.stem}.txt")

# =========================================================================
# Step 1: per-trial 32-bin LM templates (shared extraction once + mass fit per 32-bin trial)
# =========================================================================
with open(config) as f:
    base_cfg = yaml.safe_load(f)
lm_outdir = PATH.Path(base_cfg['task_LM']['outdir'])

lm_cfg = copy.deepcopy(base_cfg)
lm_cfg['pathFileSE'] = base_cfg['task_LM']['pathFileSE']
lm_cfg['pathFileME'] = base_cfg['task_LM']['pathFileME']
lm_cfg['pathFileMass'] = base_cfg['task_LM']['pathFileMass']
lm_cfg['outdir'] = str(lm_outdir)
lm_cfg['suffix'] = "lm32_shared"
lm_cfg['nDeltaPhiBins'] = 32
lm_cfg['deltaEtaBins'] = [[-1.3, -0.2], [0.2, 1.3]]
lm_cfg['task_LM']['do'] = False
lm32_cfg_path = lm_outdir / "config" / "config_lm32_shared.yaml"
lm32_cfg_path.parent.mkdir(parents=True, exist_ok=True)
with open(lm32_cfg_path, 'w') as f:
    yaml.dump(lm_cfg, f, default_flow_style=False)
LM32_shared_CR = lm_outdir / "CorrelExtract_lm32_shared" / "CorrelationsResults" / "CorrelationsResults.root"
if not LM32_shared_CR.exists():
    run_cmd(f"cd {PROJECT_ROOT}/correlations/PostProcessing && python3 {PROJECT_ROOT}/correlations/PostProcessing/ExtractOutputCorrel.py {lm32_cfg_path}",
            lm32_cfg_path.parent / "log_extract_lm32_shared.txt")
if not LM32_shared_CR.exists():
    raise RuntimeError(f"LM 32-bin CR not produced: {LM32_shared_CR}")
lm_invmass = lm_outdir / "InvMass" / "InvMassVsPt.root"
if not lm_invmass.exists():
    raise RuntimeError(f"LM InvMassVsPt.root missing: {lm_invmass}")

lm32_pair = {}   # 32-bin trial code -> its 32-bin LM PairYieldsVsPhi.root
for tid, tcfg in config_trials.items():
    if not tid.endswith("2"):
        continue
    with open(tcfg) as f:
        tc = yaml.safe_load(f)
    lm32_suffix = f"lm32_{tid}"
    dst = lm_outdir / f"CorrelExtract_{lm32_suffix}" / "AssociatedPairsYields" / "PairYieldsVsPhi.root"
    if not dst.exists():
        lm32 = copy.deepcopy(tc)
        lm32['pathFileSE'] = tc['task_LM']['pathFileSE']
        lm32['pathFileME'] = tc['task_LM']['pathFileME']
        lm32['pathFileMass'] = tc['task_LM']['pathFileMass']
        lm32['outdir'] = str(lm_outdir)
        lm32['suffix'] = lm32_suffix
        lm32['nDeltaPhiBins'] = 32
        lm32['deltaEtaBins'] = [[-1.3, -0.2], [0.2, 1.3]]
        lm32['task_LM']['do'] = False
        lm32_cfg = lm_outdir / "config" / f"config_{lm32_suffix}.yaml"
        lm32_cfg.parent.mkdir(parents=True, exist_ok=True)
        with open(lm32_cfg, 'w') as f:
            yaml.dump(lm32, f, default_flow_style=False)
        cr_dst = lm_outdir / f"CorrelExtract_{lm32_suffix}" / "CorrelationsResults" / "CorrelationsResults.root"
        cr_dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(LM32_shared_CR, cr_dst)
        run_cmd(f"cd {PROJECT_ROOT}/src && python3 {PROJECT_ROOT}/src/ry_interface.py {lm32_cfg}",
                lm32_cfg.parent / f"log_fit_mass_{lm32_suffix}.txt")
    if dst.exists():
        lm32_pair[tid] = dst
    else:
        print(f"  [WARN] 32-bin LM mass fit failed for {tid}")
print(f"[LM32] {len(lm32_pair)} per-trial 32-bin LM templates ready")

# =========================================================================
# Step 2: for each trial, run the (HM, LM) combinations with the general scale rule
# =========================================================================
for tid, tcfg in tqdm(config_trials.items(), desc="HM/LM combos", unit="trial"):
    with open(tcfg) as f:
        tc = yaml.safe_load(f)
    hm = int(tc.get('nDeltaPhiBins', 16))
    for lm in (16, 32):
        tag = f"H{hm}L{lm}"
        if tag not in RUN_COMBOS:
            continue
        new_suffix = f"trial_{tid}_{tag}"
        scale = lm / hm
        try:
            stage_hm_pair(tc, new_suffix)
            if lm == 16:
                lm_src = lm_outdir / f"CorrelExtract_{tc['suffix']}" / "AssociatedPairsYields" / "PairYieldsVsPhi.root"
            else:
                tid32 = tid[:-1] + "2"
                if tid32 not in lm32_pair:
                    print(f"  [WARN] 32-bin LM for {tid32} missing, skip {new_suffix}")
                    continue
                lm_src = lm32_pair[tid32]
            if not lm_src.exists():
                print(f"  [WARN] LM template missing: {lm_src}, skip")
                continue
            lm_dst = lm_outdir / f"CorrelExtract_{new_suffix}" / "AssociatedPairsYields" / "PairYieldsVsPhi.root"
            lm_dst.parent.mkdir(parents=True, exist_ok=True)
            scale_pairyields(lm_src, lm_dst, scale=scale)   # scale=1 -> plain copy
            vcfg = write_variant_config(tc, new_suffix)
            print(f"\n[combo] trial {tid} -> {new_suffix}  (LM scale={scale})")
            fit_correl_only(vcfg)
        except Exception as e:
            print(f"  [combo] ERROR {tid} {new_suffix}: {e}")

print("[INFO] HM/LM combination pipeline done!")


In [ ]:
# ===== Group: central-config 32-bin LM reused for every 32-bin trial's HM =====
# suffix: trial_LM_<id>_32
# HM = trial's own 32-bin mass-fit result (trial fitConfig)
# LM = ONE 32-bin template from the CENTRAL config (central fitConfig), shared by all trials
import ROOT
ROOT.gROOT.SetBatch(True)

with open(config) as f:
    base_cfg = yaml.safe_load(f)
lm_central_outdir = PATH.Path(base_cfg['task_LM']['outdir'])   # central LM dir (.../sys/LM)

# --- central-32 LM config: central fitConfig + nDeltaPhiBins=32 + deltaEta edges 0.2/1.3 ---
lm_cfg = copy.deepcopy(base_cfg)
lm_cfg['pathFileSE'] = base_cfg['task_LM']['pathFileSE']
lm_cfg['pathFileME'] = base_cfg['task_LM']['pathFileME']
lm_cfg['pathFileMass'] = base_cfg['task_LM']['pathFileMass']
lm_cfg['outdir'] = str(lm_central_outdir)
lm_cfg['suffix'] = "lm32_central"
lm_cfg['nDeltaPhiBins'] = 32
lm_cfg['deltaEtaBins'] = [[-1.3, -0.2], [0.2, 1.3]]
lm_cfg['task_LM']['do'] = False
lm32_cfg_path = lm_central_outdir / "config" / "config_lm32_central.yaml"
lm32_cfg_path.parent.mkdir(parents=True, exist_ok=True)
with open(lm32_cfg_path, 'w') as f:
    yaml.dump(lm_cfg, f, default_flow_style=False)

# extract (once, idempotent)
central_CR = lm_central_outdir / "CorrelExtract_lm32_central" / "CorrelationsResults" / "CorrelationsResults.root"
if not central_CR.exists():
    run_cmd(f"cd {PROJECT_ROOT}/correlations/PostProcessing && python3 {PROJECT_ROOT}/correlations/PostProcessing/ExtractOutputCorrel.py {lm32_cfg_path}",
            lm32_cfg_path.parent / "log_extract_lm32_central.txt")
if not central_CR.exists():
    raise RuntimeError(f"central 32-bin LM CR not produced: {central_CR}")

# mass fit (once, idempotent) -> central 32-bin LM template
central_LM_pair = lm_central_outdir / "CorrelExtract_lm32_central" / "AssociatedPairsYields" / "PairYieldsVsPhi.root"
if not central_LM_pair.exists():
    run_cmd(f"cd {PROJECT_ROOT}/src && python3 {PROJECT_ROOT}/src/ry_interface.py {lm32_cfg_path}",
            lm32_cfg_path.parent / "log_fit_mass_lm32_central.txt")
if not central_LM_pair.exists():
    raise RuntimeError(f"central 32-bin LM template not produced: {central_LM_pair}")
print(f"[central32-LM] template ready: {central_LM_pair}")

trial_ids_32 = [tid for tid in config_trials if tid.endswith("2")]

for tid in tqdm(trial_ids_32, desc="central32-LM", unit="trial"):
    with open(config_trials[tid]) as f:
        tc = yaml.safe_load(f)
    new_suffix = f"trial_LM_{tid}_32"
    try:
        stage_hm_pair(tc, new_suffix)
        lm_dst = PATH.Path(tc['task_LM']['outdir']) / f"CorrelExtract_{new_suffix}" / "AssociatedPairsYields" / "PairYieldsVsPhi.root"
        lm_dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(central_LM_pair, lm_dst)
        vcfg = write_variant_config(tc, new_suffix)
        print(f"\n[central32-LM] trial {tid} -> {new_suffix}")
        fit_correl_only(vcfg)
    except Exception as e:
        print(f"  [central32-LM] ERROR {tid}: {e}")

print("[INFO] central-config 32-bin LM group done!")


In [5]:
# ===== produce final results + prompt v2 for the _15 / _32 variant trials =====
# Self-contained: re-points the results root (old -> ROOT_OUT) and corrects each
# variant config's outdir, then runs produce_final_results + prompt v2.
import glob
import ROOT
ROOT.gROOT.SetBatch(True)

ROOT_OUT     = "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth"
PROJECT_ROOT = "/home/wuct/ALICE/reps/hf-vn-dev/dev"

ROOT_OUT = str(PATH.Path(ROOT_OUT)).rstrip("/")

# --- detect the stale results root (prefix up to "/fifth") and the sub-path after it ---
with open(config) as f:
    base = yaml.safe_load(f)
s = base.get('sys_outdir', '')
i = s.rfind('/fifth')
assert i >= 0, f"[reset] cannot detect old root from sys_outdir={s!r}"
old_root = s[:i + len('/fifth')]
sub = s[i + len('/fifth'):]          # everything after the root (identical in old & new)

if old_root != ROOT_OUT:
    p = PATH.Path(config)
    p.write_text(p.read_text().replace(old_root, ROOT_OUT))
    print(f"[reset] {old_root}\n      -> {ROOT_OUT}")
else:
    print(f"[reset] already under {ROOT_OUT}")

def _run_cmd(cmd, log):
    log.parent.mkdir(parents=True, exist_ok=True)
    print(f"  [RUN] {cmd} > {log} 2>&1")
    rc = os.system(f"{cmd} > {log} 2>&1")
    if rc != 0:
        print(f"  [WARN] exit code {rc}")
    return rc

def produce_final_results(config_path):
    with open(config_path) as f:
        cfg = yaml.safe_load(f)
    outdir_str = str(cfg['outdir'])
    j = outdir_str.rfind('/fifth')
    if j >= 0:
        outdir_str = outdir_str.replace(outdir_str[:j + len('/fifth')], ROOT_OUT)
    outdir = PATH.Path(outdir_str)
    suffix = cfg['suffix']
    input_file = (outdir / f"CorrelExtract_{suffix}" / "CorrelationFitResults"
                  / "Output_CorrelationFitting_Root" / "CorrPhiD0_FinalPlots.root")
    output_file = outdir / f"CorrelExtract_{suffix}" / "final_results.root"
    if not input_file.exists():
        print(f"  [WARN] input missing: {input_file}")
        return None
    f = ROOT.TFile.Open(str(input_file))
    for k in f.GetListOfKeys():
        h = f.Get(k.GetName())
        h.SetDirectory(0)
        h.Scale(1 / 0.07)
        o = ROOT.TFile.Open(str(output_file), "UPDATE")
        o.cd()
        h.Write(k.GetName(), ROOT.TObject.kOverwrite)
        o.Close()
    f.Close()
    print(f"  [FinalResults] {output_file}")
    return output_file

def calculate_prompt_v2(final_results_file):
    prompt_config = f"{PROJECT_ROOT}/configs/v2_prompt_v2_method_check.yml"
    _run_cmd(f"cd {PROJECT_ROOT}/src && python3 {PROJECT_ROOT}/src/compute_prompt_v2_unfold.py "
             f"{prompt_config} {final_results_file} --outpath {PATH.Path(final_results_file).parent}",
             PATH.Path(final_results_file).parent / "log_compute_prompt_v2.txt")

# locate variant configs in BOTH old and new locations (deduped)
cfg_dirs = {str(PATH.Path(old_root + sub) / "sys_fit" / "config"),
            str(PATH.Path(ROOT_OUT + sub) / "sys_fit" / "config")}
variant_configs = []
for d in cfg_dirs:
    for pat in ("config_trial_*_H*L*.yaml", "config_trial_*_15.yaml", "config_trial_*_32.yaml"):
        variant_configs += sorted(glob.glob(os.path.join(d, pat)))
variant_configs = sorted(set(variant_configs))
print(f"[INFO] {len(variant_configs)} _15/_32 variant configs")

for vcfg in tqdm(variant_configs, desc="final+prompt", unit="trial"):
    final = produce_final_results(vcfg)
    if final is not None:
        calculate_prompt_v2(final)

print("[INFO] done")


[reset] already under /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth
[INFO] 379 _15/_32 variant configs


final+prompt:   0%|          | 0/379 [00:00<?, ?trial/s]

  [FinalResults] /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/k60100_loose2to2d5_d20/sys/sys_fit/CorrelExtract_trial_11112_15/final_results.root
  [RUN] cd /home/wuct/ALICE/reps/hf-vn-dev/dev/src && python3 /home/wuct/ALICE/reps/hf-vn-dev/dev/src/compute_prompt_v2_unfold.py /home/wuct/ALICE/reps/hf-vn-dev/dev/configs/v2_prompt_v2_method_check.yml /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/k60100_loose2to2d5_d20/sys/sys_fit/CorrelExtract_trial_11112_15/final_results.root --outpath /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/k60100_loose2to2d5_d20/sys/sys_fit/CorrelExtract_trial_11112_15 > /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/k60100_loose2to2d5_d20/sys/sys_fit/CorrelExtract_trial_11112_15/log_compute_prompt_v2.txt 2>&1
  [FinalResults] /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/k60100_loose2to2d5_d20/sys/sys_fit/CorrelExtract_trial_11112_32/final_results.root
  [RUN]